# Lab 2 – Task 2.1: extract, transform and inspect
Run from the project root (the notebook changes directory to it). Extractors only download and save raw files; transforms turn a saved file into the normalized table `country_iso3, indicator_code, year, sex, value, lower, upper`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(os.getcwd())

import pandas as pd
from etl.extract_ihme import locate_ihme_csv
from etl.extract_worldbank import download_worldbank_indicator, download_worldbank_countries
from etl.transform_ihme import transform_ihme_file, transform_ihme_locations
from etl.transform_worldbank import transform_worldbank_file, transform_worldbank_countries

## 1. IHME (GBD 2023, endometriosis)

In [ ]:
ihme_path, ihme_url = locate_ihme_csv()
print(ihme_path, ihme_url)
ihme = transform_ihme_file(ihme_path)   # raises ValueError if a location name has no ISO3 code
ihme.head()

In [ ]:
ihme.info()

In [ ]:
ihme.isna().sum()

In [ ]:
ihme_loc = transform_ihme_locations(ihme_path)
print(len(ihme_loc), 'locations')
ihme_loc.head()

## 2. World Bank

In [ ]:
wb_countries_path, _ = download_worldbank_countries()
wb_countries = transform_worldbank_countries(wb_countries_path)
print(len(wb_countries), 'countries (aggregates dropped)')
wb_countries.head()

In [ ]:
WB_CODES = ['NY.GDP.PCAP.CD', 'SH.XPD.CHEX.GD.ZS', 'SH.XPD.CHEX.PC.CD', 'SP.DYN.TFRT.IN']
wb_frames = {}
for code in WB_CODES:
    path, url = download_worldbank_indicator(code)
    wb_frames[code] = transform_worldbank_file(path, code)
    print(code, path.name, len(wb_frames[code]))
wb = pd.concat(wb_frames.values(), ignore_index=True)
wb.head()

In [ ]:
wb.info()

In [ ]:
wb.isna().sum()

## 3. Share of missing values per indicator

In [ ]:
share_missing = (
    pd.concat([ihme, wb])
    .assign(value_missing=lambda d: d['value'].isna())
    .groupby('indicator_code')['value_missing']
    .agg(rows='size', missing='sum', share='mean')
)
share_missing['share'] = (share_missing['share'] * 100).round(1)
share_missing

### Notes
*(fill in after running: which indicators have the most missing values and why.)*